In [ ]:
# Notebook baseline-only: training και evaluation του τελικού baseline μοντέλου
# Δεν περιλαμβάνει pruning ούτε hyperparameter grid search
import copy
import random
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import qmc
from tqdm.auto import tqdm

# Σταθερό seed για να μπορώ να ξανατρέχω το ίδιο πείραμα
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Συσκευή εκτέλεσης
try:
    import torch_directml
    device = torch_directml.device()
    print("Χρήση AMD GPU μέσω DirectML")
except ImportError:
    device = torch.device("cpu")
    print("Χρήση CPU")

print(f"Device selected: {device}")


def derive_lbfgs_max_eval(max_iter, ratio=1.25):
    # Κρατάω το ίδιο rule-of-thumb που χρησιμοποιεί και ο κώδικας στα .py
    return max(1, int(ratio * max_iter))


# Όρια του προβλήματος
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0

# Sampling, αρχιτεκτονική και baseline hyperparameters
N_f = 10000
N_i = 1000
N_b = 500
layer_sizes = [3, 20, 20, 20, 20, 20, 20, 20, 20, 1]

epochs_adam = 500
adam_lr = 1e-3
lambda_f = 1.0
lambda_u = 100.0
lbfgs_lr = 1.0
lbfgs_max_iter = 250
lbfgs_max_eval = derive_lbfgs_max_eval(lbfgs_max_iter)

snapshot_times = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
slice_times = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
fixed_y = 0.5
N_plot = 100
N_slice = 200
N_global_space = 80
N_global_times = 11

# Ονόματα και φάκελοι εξόδου του baseline run, όπως και στα scripts
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
BASELINE_RUN_STEM = f"baseline_{epochs_adam}_{lbfgs_max_iter}-{lbfgs_max_eval}_{lambda_f:g}.{lambda_u:g}"
BASELINE_RUN_NAME = f"{BASELINE_RUN_STEM}_{RUN_TIMESTAMP}"

RUNS_DIR = Path("runs")
RUN_DIR = RUNS_DIR / BASELINE_RUN_NAME
RUN_MODELS_DIR = RUN_DIR / "models"
RUN_REPORTS_DIR = RUN_DIR / "reports"
RUN_FIGURES_DIR = RUN_DIR / "figures"

for folder in [RUN_MODELS_DIR, RUN_REPORTS_DIR, RUN_FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ADAM_PATH = RUN_MODELS_DIR / "baseline_best_adam.pth"
FINAL_PATH = RUN_MODELS_DIR / "baseline_final_lbfgs.pth"
SNAPSHOT_REPORT_PATH = RUN_REPORTS_DIR / "baseline_snapshot_metrics.xlsx"
GLOBAL_REPORT_PATH = RUN_REPORTS_DIR / "baseline_global_summary.xlsx"


def resolve_latest_adam_path():
    candidates = sorted(RUNS_DIR.glob(f"{BASELINE_RUN_STEM}_*/models/baseline_best_adam.pth"))
    if candidates:
        return candidates[-1]
    if ADAM_PATH.exists():
        return ADAM_PATH
    raise FileNotFoundError("Δεν βρέθηκε Adam checkpoint. Τρέξε πρώτα τα training cells.")


def resolve_latest_final_path():
    candidates = sorted(RUNS_DIR.glob(f"{BASELINE_RUN_STEM}_*/models/baseline_final_lbfgs.pth"))
    if candidates:
        return candidates[-1]
    if FINAL_PATH.exists():
        return FINAL_PATH
    raise FileNotFoundError("Δεν βρέθηκε τελικό L-BFGS checkpoint. Τρέξε πρώτα τα training cells.")


# Ακριβής λύση για τη 2D εξίσωση θερμότητας στο συγκεκριμένο benchmark
# u_t - (u_xx + u_yy) = 0, (x, y) ∈ [-1, 1]^2, t ∈ [0, 1]
# Αρχική συνθήκη: u(0, x, y) = sin(pi (x + 1) / 2) sin(pi (y + 1) / 2)
# Συνοριακή συνθήκη: u = 0 στα όρια του τετραγώνου
def exact_solution(t, x, y):
    return (
        torch.exp(-0.5 * np.pi**2 * t)
        * torch.sin(0.5 * np.pi * (x + 1.0))
        * torch.sin(0.5 * np.pi * (y + 1.0))
    )


In [ ]:
# Σημεία collocation με Latin Hypercube Sampling μέσα στο χωροχρονικό πεδίο
lhs_sampler = qmc.LatinHypercube(d=3, seed=SEED)
collocation = qmc.scale(
    lhs_sampler.random(n=N_f),
    [t_min, x_min, y_min],
    [t_max, x_max, y_max],
)
collocation_tensor = torch.tensor(collocation, dtype=torch.float32, device=device)
t_f = collocation_tensor[:, 0:1]
x_f = collocation_tensor[:, 1:2]
y_f = collocation_tensor[:, 2:3]

t_f.requires_grad_(True)
x_f.requires_grad_(True)
y_f.requires_grad_(True)

# Σημεία αρχικής συνθήκης: t = 0
t_i = t_min * torch.ones(N_i, 1, device=device)
x_i = (x_max - x_min) * torch.rand(N_i, 1, device=device) + x_min
y_i = (y_max - y_min) * torch.rand(N_i, 1, device=device) + y_min
u_i = exact_solution(t_i, x_i, y_i)

# Σημεία συνοριακών συνθηκών στις τέσσερις πλευρές
t_b1 = (t_max - t_min) * torch.rand(N_b, 1, device=device) + t_min
x_b1 = x_min * torch.ones(N_b, 1, device=device)
y_b1 = (y_max - y_min) * torch.rand(N_b, 1, device=device) + y_min
u_b1 = exact_solution(t_b1, x_b1, y_b1)

t_b2 = (t_max - t_min) * torch.rand(N_b, 1, device=device) + t_min
x_b2 = x_max * torch.ones(N_b, 1, device=device)
y_b2 = (y_max - y_min) * torch.rand(N_b, 1, device=device) + y_min
u_b2 = exact_solution(t_b2, x_b2, y_b2)

t_b3 = (t_max - t_min) * torch.rand(N_b, 1, device=device) + t_min
x_b3 = (x_max - x_min) * torch.rand(N_b, 1, device=device) + x_min
y_b3 = y_min * torch.ones(N_b, 1, device=device)
u_b3 = exact_solution(t_b3, x_b3, y_b3)

t_b4 = (t_max - t_min) * torch.rand(N_b, 1, device=device) + t_min
x_b4 = (x_max - x_min) * torch.rand(N_b, 1, device=device) + x_min
y_b4 = y_max * torch.ones(N_b, 1, device=device)
u_b4 = exact_solution(t_b4, x_b4, y_b4)

# Ενιαίο supervised σύνολο για αρχική και συνοριακές συνθήκες
t_u = torch.cat([t_i, t_b1, t_b2, t_b3, t_b4], dim=0)
x_u = torch.cat([x_i, x_b1, x_b2, x_b3, x_b4], dim=0)
y_u = torch.cat([y_i, y_b1, y_b2, y_b3, y_b4], dim=0)
u_real = torch.cat([u_i, u_b1, u_b2, u_b3, u_b4], dim=0)

print(f"Training data generated on {device}")
print(f" - Collocation points: {t_f.shape[0]}")
print(f" - Initial points: {t_i.shape[0]}")
print(f" - Boundary points total: {4 * N_b}")
print(f" - Supervised points total: {t_u.shape[0]}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    x_f.detach().cpu().numpy(),
    y_f.detach().cpu().numpy(),
    s=1,
    alpha=0.35,
    label="Collocation points",
)
ax.scatter(
    x_u.detach().cpu().numpy(),
    y_u.detach().cpu().numpy(),
    s=6,
    c="red",
    alpha=0.75,
    label="Initial / Boundary points",
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Training points distribution (spatial projection)")
ax.set_xlim(x_min - 0.05, x_max + 0.05)
ax.set_ylim(y_min - 0.05, y_max + 0.05)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(RUN_FIGURES_DIR / "training_points.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Το PINN που χρησιμοποιώ και στα αρχεία του package
class HeatPINN(nn.Module):
    def __init__(self, layers, t_min=0.0, t_max=1.0, x_min=-1.0, x_max=1.0, y_min=-1.0, y_max=1.0):
        super().__init__()
        self.activation = nn.Tanh()
        self.t_min = float(t_min)
        self.t_max = float(t_max)
        self.x_min = float(x_min)
        self.x_max = float(x_max)
        self.y_min = float(y_min)
        self.y_max = float(y_max)
        self.layers = nn.ModuleList(
            nn.Linear(layers[i], layers[i + 1])
            for i in range(len(layers) - 1)
        )

    @staticmethod
    def _normalize(values, lower, upper):
        return 2.0 * (values - lower) / (upper - lower) - 1.0

    def forward(self, t, x, y):
        z = torch.cat(
            [
                self._normalize(t, self.t_min, self.t_max),
                self._normalize(x, self.x_min, self.x_max),
                self._normalize(y, self.y_min, self.y_max),
            ],
            dim=1,
        )

        for layer in self.layers[:-1]:
            z = self.activation(layer(z))

        return self.layers[-1](z)


model = HeatPINN(
    layer_sizes,
    t_min=t_min,
    t_max=t_max,
    x_min=x_min,
    x_max=x_max,
    y_min=y_min,
    y_max=y_max,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=adam_lr)

print(model)


In [ ]:
# Residual της PDE: u_t - u_xx - u_yy
def pde_residual(model, t, x, y):
    u = model(t, x, y)

    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]

    return u_t - u_xx - u_yy


def physics_loss_function(model, t, x, y):
    return torch.mean(pde_residual(model, t, x, y) ** 2)


print("Το residual της PDE ορίστηκε σωστά.")


In [ ]:
# Μικρός έλεγχος ότι τα shapes είναι σωστά πριν από το training
u_test = model(t_f[:5], x_f[:5], y_f[:5])
f_test = pde_residual(model, t_f[:5], x_f[:5], y_f[:5])

print("u_test shape:", u_test.shape)
print("f_test shape:", f_test.shape)


In [ ]:
# Εκπαίδευση με Adam και αποθήκευση του καλύτερου checkpoint
history = {"total": [], "physics": [], "data": []}
best_loss = float("inf")
best_state_dict = None
best_epoch = 0

print("Ξεκινά η εκπαίδευση με Adam...")
model.train()
pbar = tqdm(range(1, epochs_adam + 1), desc="Adam")

for epoch in pbar:
    optimizer.zero_grad()

    loss_physics = physics_loss_function(model, t_f, x_f, y_f)
    u_pred = model(t_u, x_u, y_u)
    loss_data = torch.mean((u_pred - u_real) ** 2)
    loss = lambda_f * loss_physics + lambda_u * loss_data

    loss.backward()
    optimizer.step()

    loss_value = loss.item()
    loss_physics_value = loss_physics.item()
    loss_data_value = loss_data.item()

    history["total"].append(loss_value)
    history["physics"].append(loss_physics_value)
    history["data"].append(loss_data_value)

    if loss_value < best_loss:
        best_loss = loss_value
        best_epoch = epoch
        best_state_dict = copy.deepcopy(model.state_dict())

    if epoch % 100 == 0:
        pbar.set_postfix(
            total=f"{loss_value:.3e}",
            phys=f"{loss_physics_value:.3e}",
            data=f"{loss_data_value:.3e}",
        )

if best_state_dict is None:
    raise RuntimeError("Το Adam training δεν έδωσε έγκυρο state.")

model.load_state_dict(best_state_dict)
torch.save(model.state_dict(), ADAM_PATH)

print("Ολοκληρώθηκε το Adam training.")
print(f"Best Adam loss: {best_loss:.6e} at epoch {best_epoch}")
print(f"Saved Adam checkpoint to: {ADAM_PATH}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(history["total"], label="Total loss")
ax.semilogy(history["physics"], label="Physics loss")
ax.semilogy(history["data"], label="Data loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")
ax.set_title("Adam training history")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(RUN_FIGURES_DIR / "baseline_best_adam_history.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Fine-tuning με L-BFGS και αποθήκευση του καλύτερου τελικού checkpoint
print("Ξεκινά το fine-tuning με L-BFGS...")

model.load_state_dict(torch.load(resolve_latest_adam_path(), map_location="cpu"))
model.to(device)

optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=lbfgs_lr,
    max_iter=lbfgs_max_iter,
    max_eval=lbfgs_max_eval,
    history_size=50,
    tolerance_grad=1e-7,
    tolerance_change=1.0 * np.finfo(float).eps,
    line_search_fn="strong_wolfe",
)

lbfgs_history = {"total": [], "physics": [], "data": []}
tracker = {"best_loss": float("inf"), "best_step": 0, "best_state_dict": None}
closure_calls = {"count": 0}
pbar = tqdm(total=lbfgs_max_eval, desc="L-BFGS")


def closure():
    optimizer_lbfgs.zero_grad()

    loss_physics = physics_loss_function(model, t_f, x_f, y_f)
    u_pred = model(t_u, x_u, y_u)
    loss_data = torch.mean((u_pred - u_real) ** 2)
    loss = lambda_f * loss_physics + lambda_u * loss_data
    loss.backward()

    loss_value = loss.item()
    lbfgs_history["total"].append(loss_value)
    lbfgs_history["physics"].append(loss_physics.item())
    lbfgs_history["data"].append(loss_data.item())

    closure_calls["count"] += 1
    if loss_value < tracker["best_loss"]:
        tracker["best_loss"] = loss_value
        tracker["best_step"] = closure_calls["count"]
        tracker["best_state_dict"] = copy.deepcopy(model.state_dict())

    if closure_calls["count"] > pbar.total:
        pbar.total = closure_calls["count"]
    pbar.update(1)
    pbar.set_postfix(
        total=f"{loss_value:.3e}",
        phys=f"{loss_physics.item():.3e}",
        data=f"{loss_data.item():.3e}",
    )

    return loss


model.train()
try:
    optimizer_lbfgs.step(closure)
finally:
    pbar.close()

if tracker["best_state_dict"] is None:
    raise RuntimeError("Το L-BFGS δεν έδωσε έγκυρο state.")

model.load_state_dict(tracker["best_state_dict"])
torch.save(model.state_dict(), FINAL_PATH)

print("Ολοκληρώθηκε το L-BFGS fine-tuning.")
print(f"Best L-BFGS loss: {tracker['best_loss']:.6e} at evaluation {tracker['best_step']}")
print(f"Saved final checkpoint to: {FINAL_PATH}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(lbfgs_history["total"], label="Total loss")
ax.semilogy(lbfgs_history["physics"], label="Physics loss")
ax.semilogy(lbfgs_history["data"], label="Data loss")
ax.set_xlabel("L-BFGS evaluation")
ax.set_ylabel("Loss (log scale)")
ax.set_title("L-BFGS fine-tuning history")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(RUN_FIGURES_DIR / "baseline_final_lbfgs_history.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Snapshot αξιολόγηση του τελικού baseline checkpoint μετά το L-BFGS
state_dict = torch.load(resolve_latest_final_path(), map_location="cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()


def relative_l2_error(u_pred, u_exact, eps=1e-12):
    num = torch.sqrt(torch.sum((u_pred - u_exact) ** 2))
    den = torch.sqrt(torch.sum(u_exact ** 2))
    return (num / (den + eps)).item()


def mae(u_pred, u_exact):
    return torch.mean(torch.abs(u_pred - u_exact)).item()


def rmse(u_pred, u_exact):
    return torch.sqrt(torch.mean((u_pred - u_exact) ** 2)).item()


x_vals = np.linspace(x_min, x_max, N_plot)
y_vals = np.linspace(y_min, y_max, N_plot)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
x_flat = X_grid.flatten()
y_flat = Y_grid.flatten()

snapshot_metrics = []
snapshot_cache = []
fig, axes = plt.subplots(len(snapshot_times), 3, figsize=(18, 4 * len(snapshot_times)))

for fixed_t in snapshot_times:
    t_flat = np.full_like(x_flat, fixed_t)
    x_test = torch.tensor(x_flat, dtype=torch.float32).view(-1, 1).to(device)
    y_test = torch.tensor(y_flat, dtype=torch.float32).view(-1, 1).to(device)
    t_test = torch.tensor(t_flat, dtype=torch.float32).view(-1, 1).to(device)

    with torch.no_grad():
        u_pred = model(t_test, x_test, y_test)
        u_exact = exact_solution(t_test, x_test, y_test)
        abs_error = torch.abs(u_pred - u_exact)

    snapshot_metrics.append({
        "t": fixed_t,
        "max_u_exact": torch.max(torch.abs(u_exact)).item(),
        "max_u_pred": torch.max(torch.abs(u_pred)).item(),
        "relative_l2": relative_l2_error(u_pred, u_exact),
        "mae": mae(u_pred, u_exact),
        "rmse": rmse(u_pred, u_exact),
        "max_error": torch.max(abs_error).item(),
    })
    snapshot_cache.append((
        fixed_t,
        u_pred.cpu().numpy().reshape(N_plot, N_plot),
        u_exact.cpu().numpy().reshape(N_plot, N_plot),
        abs_error.cpu().numpy().reshape(N_plot, N_plot),
    ))

value_vmin = min(U_exact_grid.min() for _, _, U_exact_grid, _ in snapshot_cache)
value_vmax = max(U_exact_grid.max() for _, _, U_exact_grid, _ in snapshot_cache)
error_vmax = max(Error_grid.max() for _, _, _, Error_grid in snapshot_cache)

for i, (fixed_t, U_pred_grid, U_exact_grid, Error_grid) in enumerate(snapshot_cache):
    im1 = axes[i, 0].imshow(U_pred_grid, extent=[x_min, x_max, y_min, y_max], origin="lower", cmap="viridis", vmin=value_vmin, vmax=value_vmax)
    axes[i, 0].set_title(f"Predicted u(t={fixed_t:.2f})", fontweight="bold")
    axes[i, 0].set_xlabel("x")
    axes[i, 0].set_ylabel("y")
    plt.colorbar(im1, ax=axes[i, 0])

    im2 = axes[i, 1].imshow(U_exact_grid, extent=[x_min, x_max, y_min, y_max], origin="lower", cmap="viridis", vmin=value_vmin, vmax=value_vmax)
    axes[i, 1].set_title(f"Exact u(t={fixed_t:.2f})", fontweight="bold")
    axes[i, 1].set_xlabel("x")
    axes[i, 1].set_ylabel("y")
    plt.colorbar(im2, ax=axes[i, 1])

    im3 = axes[i, 2].imshow(Error_grid, extent=[x_min, x_max, y_min, y_max], origin="lower", cmap="viridis", vmin=0.0, vmax=max(error_vmax, 1e-12))
    axes[i, 2].set_title(f"Absolute error at t={fixed_t:.2f}", fontweight="bold")
    axes[i, 2].set_xlabel("x")
    axes[i, 2].set_ylabel("y")
    plt.colorbar(im3, ax=axes[i, 2])

plt.tight_layout()
fig.savefig(RUN_FIGURES_DIR / "baseline_final_lbfgs_snapshots.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

metrics_df = pd.DataFrame(snapshot_metrics)
metrics_df.to_excel(SNAPSHOT_REPORT_PATH, index=False)
metrics_df


In [ ]:
# Συνοπτικός πίνακας των snapshot metrics του τελικού checkpoint
metrics_df

print("Average metrics across snapshots:")
print(metrics_df[["relative_l2", "mae", "rmse", "max_error"]].mean())
print(f"Saved snapshot report to: {SNAPSHOT_REPORT_PATH}")


In [ ]:
# Σύγκριση τομών u(t, x, y=fixed_y) για πολλά χρονικά στιγμιότυπα
state_dict = torch.load(resolve_latest_final_path(), map_location="cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()

x_vals = np.linspace(x_min, x_max, N_slice)
x_torch = torch.tensor(x_vals, dtype=torch.float32).view(-1, 1).to(device)
y_torch = torch.full_like(x_torch, fixed_y).to(device)

slice_min = float("inf")
slice_max = float("-inf")
for t_val in slice_times:
    t_torch = torch.full_like(x_torch, t_val).to(device)
    with torch.no_grad():
        u_exact = exact_solution(t_torch, x_torch, y_torch).cpu().numpy()
    slice_min = min(slice_min, float(np.min(u_exact)))
    slice_max = max(slice_max, float(np.max(u_exact)))

span = slice_max - slice_min
pad = 0.05 * span if span > 0 else 0.05
y_limits = (slice_min - pad, slice_max + pad)

fig = plt.figure(figsize=(4 * len(slice_times), 5))
for i, t_val in enumerate(slice_times):
    t_torch = torch.full_like(x_torch, t_val).to(device)
    with torch.no_grad():
        u_pred = model(t_torch, x_torch, y_torch).cpu().numpy()
        u_exact = exact_solution(t_torch, x_torch, y_torch).cpu().numpy()

    ax = plt.subplot(1, len(slice_times), i + 1)
    ax.plot(x_vals, u_exact, "b-", linewidth=2, label="Exact")
    ax.plot(x_vals, u_pred, "r--", linewidth=2, label="PINN")
    ax.set_title(f"t = {t_val:.2f}", fontweight="bold")
    ax.set_xlabel("x")
    ax.set_ylim(*y_limits)
    ax.grid(True, linestyle=":", alpha=0.6)

    if i == 0:
        ax.set_ylabel(f"u(t, x, y={fixed_y})")
        ax.legend(loc="upper right")

plt.suptitle("Slice comparison: exact vs PINN", fontsize=16)
plt.tight_layout()
fig.savefig(RUN_FIGURES_DIR / "baseline_final_lbfgs_slices.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Global αξιολόγηση του τελικού baseline μοντέλου σε πυκνότερο χρονικό πλέγμα
x_vals = np.linspace(x_min, x_max, N_global_space)
y_vals = np.linspace(y_min, y_max, N_global_space)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
x_flat = X_grid.flatten()
y_flat = Y_grid.flatten()

test_times = np.linspace(t_min, t_max, N_global_times)
global_rel_l2 = []

for t_val in test_times:
    t_flat = np.full_like(x_flat, t_val)
    x_test = torch.tensor(x_flat, dtype=torch.float32).view(-1, 1).to(device)
    y_test = torch.tensor(y_flat, dtype=torch.float32).view(-1, 1).to(device)
    t_test = torch.tensor(t_flat, dtype=torch.float32).view(-1, 1).to(device)

    with torch.no_grad():
        u_pred = model(t_test, x_test, y_test)
        u_exact = exact_solution(t_test, x_test, y_test)

    global_rel_l2.append(relative_l2_error(u_pred, u_exact))

global_summary_df = pd.DataFrame([{
    "global_mean_relative_l2": float(np.mean(global_rel_l2)),
    "global_worst_relative_l2": float(np.max(global_rel_l2)),
}])
global_summary_df.to_excel(GLOBAL_REPORT_PATH, index=False)

print("Global mean Relative L2 over the full time domain:", global_summary_df.iloc[0]["global_mean_relative_l2"])
print("Worst Relative L2 over the full time domain:", global_summary_df.iloc[0]["global_worst_relative_l2"])
print(f"Saved global summary to: {GLOBAL_REPORT_PATH}")
global_summary_df
